In [22]:
import pandas as pd
import numpy as np

In [23]:
df = pd.read_csv("retail_store_sales.csv") #This line loads the CSV file into a Pandas DataFrame.
print(df.head()) #head() shows the first 5 rows of the dataset.

  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1    TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2    TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3    TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4    TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  Digital Wallet   Online       2024-04-08   
1       9.0        261.0  Digital Wallet   Online       2023-07-23   
2       2.0         43.0     Credit Card   Online       2022-10-05   
3       9.0        247.5     Credit Card   Online       2022-05-07   
4       7.0         87.5  Digital Wallet   Online       2022-10-02   

  Discount Applied  
0             True  
1             True  
2            False  
3              NaN  
4          

In [24]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB
None


In [25]:
print(df.isnull().sum()) #isnull().sum() counts missing values in every column.

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [26]:
print(df.duplicated().sum()) #This checks wheather duplicate rows exist.

0


In [27]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date']) #Currently, the date column is stored as object-we have to convert it to datetime for analysis.
print(df.dtypes) #Checking datetimes

Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied            object
dtype: object


In [28]:
#Handle Missing Values in Numerical Columns: price per unit, Quantity, Total Spent
print(df[['Price Per Unit','Quantity','Total Spent']].isnull().sum())


Price Per Unit    609
Quantity          604
Total Spent       604
dtype: int64


In [29]:
#Filling Missing Values Using Median
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Price Per Unit'].median())
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Total Spent'] = df['Total Spent'].fillna(df['Total Spent'].median())
print(df.isnull().sum())



Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit         0
Quantity               0
Total Spent            0
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [30]:
#Handle Missing Values in Categorical Columns: Item and Discount Applied
df['Item'] = df['Item'].fillna("Unknown Item") #If product name is missing, we replace it with: Unknown Item(This prevents data loss.)
df['Discount Applied'] = df['Discount Applied'].fillna('False') #Missing discount values usually mean no discount was applied. So we replace missing values with: False

In [31]:
print(df.isnull().sum())

Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


In [32]:
print(df.describe()) #Check Statistical Summary

       Price Per Unit      Quantity   Total Spent  \
count    12575.000000  12575.000000  12575.000000   
mean        23.348191      5.558648    128.636581   
min          5.000000      1.000000      5.000000   
25%         14.000000      3.000000     55.000000   
50%         23.000000      6.000000    108.500000   
75%         32.000000      8.000000    184.000000   
max         41.000000     10.000000    410.000000   
std         10.480413      2.790160     92.557580   

                    Transaction Date  
count                          12575  
mean   2023-07-12 20:23:41.105368064  
min              2022-01-01 00:00:00  
25%              2022-09-30 00:00:00  
50%              2023-07-13 00:00:00  
75%              2024-04-24 00:00:00  
max              2025-01-18 00:00:00  
std                              NaN  


In [33]:
#Detect Outliers: We use the IQR (Interquartile Range) method.
Q1 = df['Total Spent'].quantile(0.25)
Q3 = df['Total Spent'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

outliers = df[(df['Total Spent'] < lower_limit) |
              (df['Total Spent'] > upper_limit)]

print(outliers)

      Transaction ID Customer ID       Category          Item  Price Per Unit  \
27       TXN_1599706     CUST_14      Furniture   Item_25_FUR            41.0   
120      TXN_8215469     CUST_17  Milk Products  Item_23_MILK            38.0   
129      TXN_1112365     CUST_19           Food  Item_24_FOOD            39.5   
133      TXN_2953434     CUST_25      Furniture   Item_25_FUR            41.0   
135      TXN_1249742     CUST_05  Milk Products  Item_24_MILK            39.5   
...              ...         ...            ...           ...             ...   
12209    TXN_1113295     CUST_11  Milk Products  Item_25_MILK            41.0   
12216    TXN_9897293     CUST_23     Patisserie   Item_24_PAT            39.5   
12243    TXN_8227894     CUST_24      Furniture   Item_23_FUR            38.0   
12542    TXN_7484072     CUST_02       Butchers   Item_25_BUT            41.0   
12557    TXN_7239201     CUST_24      Beverages   Item_25_BEV            41.0   

       Quantity  Total Spen

In [34]:
# Create New Useful Columns
df['Year'] = df['Transaction Date'].dt.year #Extract year


In [35]:
#Extract Month
df['Month'] = df['Transaction Date'].dt.month

In [36]:
#Extract Day Name
df['Day Name'] = df['Transaction Date'].dt.day_name()

In [37]:
#Check Final Dataset
print(df.head())

  Transaction ID Customer ID       Category          Item  Price Per Unit  \
0    TXN_6867343     CUST_09     Patisserie   Item_10_PAT            18.5   
1    TXN_3731986     CUST_22  Milk Products  Item_17_MILK            29.0   
2    TXN_9303719     CUST_02       Butchers   Item_12_BUT            21.5   
3    TXN_9458126     CUST_06      Beverages   Item_16_BEV            27.5   
4    TXN_4575373     CUST_05           Food   Item_6_FOOD            12.5   

   Quantity  Total Spent  Payment Method Location Transaction Date  \
0      10.0        185.0  Digital Wallet   Online       2024-04-08   
1       9.0        261.0  Digital Wallet   Online       2023-07-23   
2       2.0         43.0     Credit Card   Online       2022-10-05   
3       9.0        247.5     Credit Card   Online       2022-05-07   
4       7.0         87.5  Digital Wallet   Online       2022-10-02   

  Discount Applied  Year  Month   Day Name  
0             True  2024      4     Monday  
1             True  2023  

In [38]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    12575 non-null  object        
 1   Customer ID       12575 non-null  object        
 2   Category          12575 non-null  object        
 3   Item              12575 non-null  object        
 4   Price Per Unit    12575 non-null  float64       
 5   Quantity          12575 non-null  float64       
 6   Total Spent       12575 non-null  float64       
 7   Payment Method    12575 non-null  object        
 8   Location          12575 non-null  object        
 9   Transaction Date  12575 non-null  datetime64[ns]
 10  Discount Applied  12575 non-null  object        
 11  Year              12575 non-null  int32         
 12  Month             12575 non-null  int32         
 13  Day Name          12575 non-null  object        
dtypes: datetime64[ns](1), 

In [39]:
#Save Cleaned Dataset
df.to_csv("cleaned_retail_store_sales.csv", index=False) #This saves the cleaned dataset into a new CSV file.index=False prevents extra index column creation.

In [40]:
print("Dataset cleaned successfully!")

Dataset cleaned successfully!
